In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import MNIST

In [2]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))   # 1 channel
])

trainset = MNIST(root="./data1",download=True,train=True,transform=transform)
testset = MNIST(root="./data1",download=True,train=False,transform=transform)

In [3]:
train_loader = DataLoader(trainset,batch_size=64,shuffle=True)
test_loader = DataLoader(testset,batch_size=64)

# Build the CNN

In [6]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers  = nn.Sequential(
            nn.Conv2d(1,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), 
    
            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
    
            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(3*3*128,256),
            nn.ReLU(),
            nn.Linear(256,10)
        )

    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0),-1) # flattening
        x = self.fc_layers(x)
        return x

In [7]:
model = CNN()

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters())

# Train the model

In [8]:
model.train()
epochs = 10
for epoch in range(epochs):
  epoch_training_loss = 0.0

  for images,labels in train_loader:
    optimizer.zero_grad()
    output = model.forward(images)
    loss = criterion(output,labels)
    loss.backward()
    optimizer.step()
    epoch_training_loss += loss.item()


  print(f"epoch : {epoch+1} & loss : {epoch_training_loss/len(train_loader)}")

epoch : 1 & loss : 0.1456157548069112
epoch : 2 & loss : 0.04046669397891602
epoch : 3 & loss : 0.02947693048644336
epoch : 4 & loss : 0.02236807618790598
epoch : 5 & loss : 0.018587607018593567
epoch : 6 & loss : 0.01402476735581267
epoch : 7 & loss : 0.013296914892639808
epoch : 8 & loss : 0.012001770853786661
epoch : 9 & loss : 0.009887635358577626
epoch : 10 & loss : 0.009283807132437193


# Evaluate the model

In [9]:
model.eval()
correct_labels = 0
total_labels = 0

with torch.no_grad():
  for images,labels in test_loader:
    outputs = model.forward(images)
    _,predicted = torch.max(outputs,1)
    correct_labels += (predicted == labels).sum().item()
    total_labels += labels.size(0)

print(f"Accuracy Score : {correct_labels/total_labels}")

Accuracy Score : 0.9904
